# Test-time Compute 缩放：训练完成后仍可投入的算力

> 第 1 讲把 Agent 定义为"一次调用之外还能做更多计算"的结构，那还是一个定性的描述。这一讲把"更多计算"变成一条可度量的缩放轴：模型训练结束后，不更新权重，只在推理时多花算力，能力还能提升多少。
>
> 我们从最简单的重复采样开始，从零实现 pass@k 无偏估计、多数投票与 best-of-n，再研究给定一笔固定算力预算时如何按题目难度分配、让收益最大，最后把多种技术组合成分层系统，用自动搜索寻找最优配置。全部实验使用合成数据，离线即可复现。


训练结束后的模型是一个固定的函数：给定一个问题，它输出一个候选答案。想让它更强，传统的路径是继续训练，但那要重跑昂贵的训练流程。这一讲关注另一条不动权重的路径——只改变推理阶段怎么花算力。这类算力统称 test-time compute，重复采样、自我修订、候选验证、多数投票都属于它。

先体会"不动权重也能变强"。假设某道题单次做对的概率是 0.3，独立尝试 k 次，至少一次做对的概率是 $1 - (1 - 0.3)^k$。k 取 1、2、5、10 时，概率分别是 0.30、0.51、0.83、0.97。同一个模型，只是多花了推理算力，一道题的通过概率就从 0.3 提升到接近 1。

这个最小实验说明，推理阶段花算力的方式能直接改变准确率。剩下的问题是这种改变有多大、预算怎么分配收益最大。第一节先建立一个合成基准，把"一道题能否被解出"变成可测量的数量。


## 1. 训练完成后的算力：test-time compute

为了给"花算力"一个统一视角，Snell 等人的工作把全部 test-time 方法归到两个旋钮上。第一个旋钮是提议分布（proposer），决定模型生成什么样的候选，重复采样、引导自我修订都改这里。第二个旋钮是输出后处理（verifier），拿到一批候选后怎么挑，best-of-n 挑选、多数投票都改这里。后面几节的每个演示都落在两个旋钮的某个组合里。

先建立贯穿全讲的合成实验环境。我们用 GSM8K 风格的数学应用题做参照：一个模型在每道题上有一个真实的单次正确率 $p@1$，记为 $p_i$。题目有难有易，$p_i$ 在题目之间呈重尾分布——大多数题很难，少量题很容易。之后的每个实验都以这批题目和它们的 $p_i$ 为输入。先手动算一遍"多试几次"的收益，再生成这批题目。


In [ ]:
import numpy as np

p = 0.3
print("单次正确率 0.3 的题，独立尝试 k 次，至少一次做对的概率")
for k in (1, 2, 5, 10, 20):
    prob = 1 - (1 - p) ** k
    print("k = %2d -> %.3f" % (k, prob))

In [ ]:
def make_benchmark(n=200, alpha=0.25, scale=0.15, seed=42):
    """生成合成基准：n 道题，每道题一个单次正确率 p@1。

    p@1 取自 Kumaraswamy(alpha, 1) 再乘 scale，密度在 0 附近
    正比于 p^(alpha-1)，形成重左尾：少量难题几乎解不出，
    少量易题很快能解。返回形状 (n,) 的数组。
    """
    rng = np.random.default_rng(seed)
    u = rng.random(n)
    return scale * u ** (1.0 / alpha)

p1 = make_benchmark()
print("题目数:", p1.shape[0])
print("p@1 均值: %.4f  中位数: %.4f" % (p1.mean(), np.median(p1)))
n = p1.shape[0]
print("最易 10%% 的平均 p@1: %.4f" % np.sort(p1)[-n // 10:].mean())
print("最难 10%% 的平均 p@1: %.4f" % np.sort(p1)[:n // 10].mean())

## 2. 重复采样：Large Language Monkeys

Large Language Monkeys 用一句话回答了第 1 节的问题：重复采样能大幅提高覆盖率。对每个问题独立采样 k 个候选解，再用验证器挑一个，这就是 `pass@k`。论文定义了配套的两个量：覆盖率（coverage）是 k 个候选里至少有一个正确的题目占比，回答"模型到底能不能解出这道题"；精度（precision）是挑出的候选里正确的比例，回答"验证器能不能把正确的候选从大量候选中挑出来"。

论文给出大量实证数字。SWE-bench Lite 上，DeepSeek-Coder-V2-Instruct 单次只解出 15.9% 的真实 GitHub issue，采 250 次后提升到 56%，超过当时的单次 SOTA 十三个百分点。Gemma-2B 在 CodeContests 上 pass@1 只有 0.02%，pass@10k 提升到 7.1%，提高了 300 倍。采样次数能稳定换取覆盖率，而且收益形态有规律可循。先处理测量口径，再看规律。



### 覆盖率与精度的口径，先用手算确定

覆盖率（coverage）和精度（precision）是一对容易混淆的指标。先用手算把两者的口径确定下来。假设给 3 道题各采样 k = 4 个候选，正确用 ✓ 标出，错误用 ✗ 标出：

| 题目 | 4 个候选 | 正确数 |
|:---|:---|:---|
| A | ✓ ✓ ✗ ✗ | 2 |
| B | ✗ ✗ ✗ ✗ | 0 |
| C | ✓ ✗ ✗ ✓ | 2 |

覆盖率按题目统计：只要一道题的候选里至少有一个 ✓，这道题就计为被覆盖。A 和 C 被覆盖，B 没有，所以覆盖率 = 2/3。它回答"模型到底能不能解出这道题"，与选择器无关。

精度按候选统计。假如选择器在 4 个候选里随机挑一个，12 个候选里共有 4 个正确，那么它挑到正确的概率是 4/12 = 1/3，这就是精度。它回答"选择器挑得准不准"。精度低不等于题目解不出：A 和 C 即使随机挑，也有机会挑中正确解。

覆盖率衡量产生候选这一端，精度衡量挑出候选这一端。同一次采样里覆盖率可以很高而精度很低，两者回答的是两个不同的问题。后面 best-of-n 的收益会写成两者的乘积，那是把两个条件叠加后的近似。


但"采样 k 次"里有一个测量细节。如果每道题生成 N 个样本、其中 C 个正确，直接报告"有正确样本的题目占比"，会系统性高估 pass@k——因为 N 个样本都被用上了，等价于 pass@N。Chen 等人给出无偏估计：

$$\mathrm{pass@k} = \frac{1}{P}\sum_{i=1}^{P}\left[1 - \frac{\binom{N-C_i}{k}}{\binom{N}{k}}\right]$$

分子是从 N 个样本里选出 k 个、一个正确的都没选到的组合数。先手算一个例子。某道题采样 N = 10 个样本，其中 C = 3 个正确。k = 1 时，$\binom{7}{1}/\binom{10}{1} = 0.7$，pass@1 = 0.3，等于 C/N，符合直觉；k = 5 时，$\binom{7}{5}/\binom{10}{5} = 21/252 \approx 0.083$，pass@5 ≈ 0.917；k = 10 时 $\binom{7}{10} = 0$，pass@10 = 1。这三个值留给下面的代码验证。


### 直接统计为什么会高估 pass@k

我们先用一个把偏差放到最大的小例子。某道题采样 N = 3 个样本，其中 C = 1 个正确，样本集写作 {✓, ✗₁, ✗₂}。要估计 pass@1，也就是只看 1 个样本就能做对的概率。三个样本里只有一个是正确的，真值是 1/3。

但直接数"这 3 个样本里有没有正确的"，答案是 1，因为样本里确实出现了 ✓。把 1 当作 pass@1 就严重高估了。原因在于：统计时把全部 N 个样本都用上了，回答的其实是 pass@N 而不是 pass@k。只要 N 个样本里出现过正确就记 1，相当于在只看 k 个的条件下把所有样本都看了一遍。

无偏估计换一个问法：从 N 个样本里随机挑 k 个，有多少比例的挑法能看到至少一个正确的？这个例子里挑 1 个样本共有 $\binom{3}{1}=3$ 种挑法，只有 {✓} 这一种能看到正确，所以 pass@1 = 1/3，和真值一致。

推广到一般情形。N 个样本里有 C 个正确，挑 k 个的方式共有 $\binom{N}{k}$ 种；一个正确都没挑到的挑法，必须从 N - C 个错误样本里挑，共 $\binom{N-C}{k}$ 种。没挑到正确的占比是 $\binom{N-C}{k}/\binom{N}{k}$，用 1 减掉，就是至少挑到一个正确的概率：

$$\mathrm{pass@k} = 1 - \frac{\binom{N-C}{k}}{\binom{N}{k}}$$

代码里 `1 - np.prod(1 - k / denom)` 是同一个式子的逐项写法，其中 `denom = arange(N-C+1, N+1)`。展开验证：$\binom{N-C}{k}/\binom{N}{k} = \frac{(N-C)!\,(N-k)!}{N!\,(N-C-k)!}$，而逐项乘积 $\prod_{d=N-C+1}^{N}(d-k)/d$ 的分子 $(N-C+1-k)\cdots(N-k) = (N-k)!/(N-C-k)!$、分母 $N!/(N-C)!$，约分后正好相等。逐项写的好处是每个因子都落在 (0, 1] 内，不会像 $\binom{N}{k}$ 那样在 N 很大时数值溢出。

还有一个边界：当 k > N - C 时，从 N - C 个错误样本里挑 k 个做不到，$\binom{N-C}{k} = 0$，估计值应为 1。逐项形式里会出现零因子 $d - k = 0$，乘积为 0，1 减 0 得到 1，自动覆盖这个边界。


In [ ]:
def estimate_pass_at_k(num_correct, num_samples, k):
    """Chen 等人的 pass@k 无偏估计，数值稳定形式。

    num_correct：num_samples 个样本里正确的个数；k：要估计的采样次数。
    等价于 1 - C(num_samples-num_correct, k) / C(num_samples, k)。
    返回 (0, 1] 的估计值。
    """
    if num_correct == 0:
        return 0.0
    denom = np.arange(num_samples - num_correct + 1, num_samples + 1)
    return 1.0 - np.prod(1.0 - k / denom)

for k in (1, 5, 10):
    print("N=10, C=3, k=%2d -> pass@k = %.4f" % (k, estimate_pass_at_k(3, 10, k)))

from math import comb

def estimate_pass_at_k_comb(num_correct, num_samples, k):
    """用组合数写出的同一估计，用于交叉验证。"""
    if k > num_samples - num_correct:
        return 1.0
    return 1.0 - comb(num_samples - num_correct, k) / comb(num_samples, k)

for k in (1, 5, 10):
    assert abs(estimate_pass_at_k(3, 10, k) - estimate_pass_at_k_comb(3, 10, k)) < 1e-12
print("数值稳定形式与组合数形式完全一致")

In [ ]:
p1 = make_benchmark()
N = 50
rng = np.random.default_rng(7)
correct = rng.binomial(N, p1)                  # 每道题 N 个样本里正确的个数

ks = np.array([1, 5, 10, 25, 50])
true_pass = np.array([(1 - (1 - p1) ** k).mean() for k in ks])
unbiased = np.array(
    [[estimate_pass_at_k(c, N, k) for c in correct] for k in ks]
).mean(axis=1)
naive = (correct >= 1).mean()                  # 只要 N 个里出现过正确就算对

print("k   true   unbiased   naive")
for i, k in enumerate(ks):
    print("%3d  %.3f    %.3f    %.3f" % (k, true_pass[i], unbiased[i], naive))
print("关键观察：k=1 时朴素估计 0.43，真值只有 0.03；"
      "朴素估计把 N 个样本全用上，等价于 pass@50，严重高估小 k。")

把合成基准的覆盖率画成 k 的函数，会得到一条典型的推理期缩放曲线。论文里 Llama-3-8B-Instruct 在 MATH 上的覆盖率先从 100 样本的 82.9% 提升到 10000 样本的 98.44%，拟合出 $\text{coverage} = \exp(a\,k^b)$，其中 a = -1.33、b = -0.43。我们在合成基准上复现这条曲线，并用最小二乘拟合同一形式的幂律。



### 幂律是什么，以及为什么用 log-log 拟合

coverage 曲线 $c(k) = \exp(a\,k^b)$ 是典型的幂律。幂律指某个量随另一个量的幂次变化，$k^b$ 就是 k 的 b 次方。它与指数衰减 $(1-p)^k$ 的区分方式是画图：指数衰减在 log-linear 图上呈直线，k 每增加固定量，值缩小固定比例；幂律在 log-log 图上呈直线，k 每放大固定倍数，值缩小固定倍数。

用论文参数 a = -1.33、b = -0.43 手算两个点。k = 100 时 $100^{-0.43} = 10^{-0.86} \approx 0.138$，$c(100) = \exp(-1.33 \times 0.138) = e^{-0.18} \approx 0.83$，对应论文的 82.9%。k = 10000 时 $10000^{-0.43} = 10^{-1.72} \approx 0.019$，$c(10000) = e^{-0.025} \approx 0.97$，与论文的 98.44% 同量级（拟合值本身是对观测的近似）。曲线快速爬升后趋于饱和。为什么聚合覆盖率恰好呈幂律？Schaeffer 等人在 《How Do Large Language Monkeys Get Their Power (Laws)?》里给出了解释：单道题的失败率是指数衰减，但当各题的 pass@1 呈重左尾分布时，聚合到整个基准的失败率就从指数衰减变成幂律。下面一节手算这个转变。

要对数据拟合 $c = \exp(a k^b)$，先把方程变形。两边取对数得 $\log c = a k^b$。a 是负数，$\log c$ 也是负数，不方便再取对数，于是先取负号：$-\log c = -a k^b$，再取对数：

$$\log(-\log c) = \log(-a) + b\log k$$

右边是 $\log k$ 的一次函数，斜率就是 b，截距是 $\log(-a)$。因此在 $\log(-\log c)$ 对 $\log k$ 的图上，数据落在一条直线上，用一次线性回归（`np.polyfit`，degree 1）拟合斜率与截距，就能还原出 b 和 a。代码里 `fit_power_law` 做的就是这件事。


In [ ]:
def coverage_at_k(p1, k):
    """合成基准的覆盖率：k 个样本里至少一个正确的题目占比。"""
    return 1 - (1 - p1) ** k

ks = np.logspace(1, 2.7, 40).astype(int)       # k = 10 .. 500，对数 40 点
covs = np.array([coverage_at_k(p1, k).mean() for k in ks])

def fit_power_law(ks, covs):
    """拟合 coverage = exp(a * k^b)。

    对 log(-log covs) 关于 log ks 做线性回归：斜率即 b，截距即 log(-a)。
    要求 0 < covs < 1。
    """
    slope, intercept = np.polyfit(np.log(ks), np.log(-np.log(covs)), 1)
    return -np.exp(intercept), slope

a, b = fit_power_law(ks, covs)
pred = np.exp(a * ks ** b)
relerr = np.abs(pred - covs) / covs
print("拟合 coverage = exp(a * k^b): a = %.3f, b = %.3f" % (a, b))
print("平均相对误差: %.4f" % relerr.mean())

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(ks, covs, "o-", label="simulated coverage")
ax.loglog(ks, pred, "--", label="fit exp(a*k^b)")
ax.set_xlabel("k (samples per problem)")
ax.set_ylabel("coverage")
ax.set_title("Coverage scaling on synthetic benchmark")
ax.legend()
plt.show()
print("关键观察：log-log 上近似直线，平均相对误差约 %.1f%%，重现论文的幂律形态。"
      % (100 * relerr.mean()))

拟合出来的幂律 $\exp(a\,k^b)$ 有一个出人意料的来源。单独看一道题，失败率是 $(1-p_i)^k$，随 k 指数衰减。但把整个基准合起来看，聚合失败率只按幂律缓慢下降。同一个采样过程里，单题指数衰减与聚合幂律并存，原因在于 $p@1$ 在题目之间的分布是重尾的。Monkey Power Laws 一文的定理 3.1 证明，如果 $p@1$ 的密度在 0 附近像 $C\,p^{b-1}$ 那样发散，聚合失败率就按 $k^{-b}$ 缩放；反过来，聚合呈幂律也要求 $p@1$ 分布有这种重左尾。少量几乎解不出的题把整体曲线拖成了幂律。我们同时画出单题与聚合两条失败率曲线，再移除重尾分布，看幂律是否消失。



### 聚合失败率从指数衰减变成幂律的推导

先看单独一道题。p@1 = p 固定时，失败率是 $(1-p)^k$，这是指数衰减：k 每增加一个常数，失败率缩小固定比例。例如 p = 0.01 的题，k 从 1 到 1000，失败率从 0.99 降到 $0.99^{1000} \approx 4.3\times10^{-5}$，下降约两万倍。

整个基准合起来时，失败率变成 $F(k) = \frac{1}{P}\sum_i(1-p_i)^k = \mathbb{E}[(1-p)^k]$。如果 p 在所有题目上是同一个值，这个期望就是单个指数衰减；当 p 在题目之间呈重左尾分布，也就是大量题目堆在 p 接近 0 的地方、只有少量题目 p 稍大时，期望的衰减就被那些接近解不出的题目拖慢。

把重左尾具体化。设 p 的密度在 0 附近正比于 $C\,p^{b-1}$（b < 1 时它在 0 附近发散）。用连续化近似 $F(k) = \int_0^1 (1-p)^k f(p)\,dp$。k 很大时 $(1-p)^k \approx e^{-kp}$，且积分主要由 p 很小的区域贡献。换元 $u = kp$，$p = u/k$，$dp = du/k$：

$$F(k) \approx \int C\left(\frac{u}{k}\right)^{b-1} e^{-u}\,\frac{du}{k} = C\,k^{-b}\int_0^{\infty} u^{b-1} e^{-u}\,du$$

$\int_0^{\infty} u^{b-1} e^{-u}\,du$ 是只依赖 b 的常数（伽马函数），所以 $F(k) \propto k^{-b}$。这就是聚合失败率呈幂律的来源：单题的指数衰减被重尾分布平均掉，剩下的主导项是幂次。定理 3.1 陈述的正是这个关系，并且反过来也成立——观测到聚合幂律，说明 p@1 分布具有这种重左尾。

合成基准 alpha = 0.25 对应密度 $f(p) \propto p^{-0.75}$，发散程度高，理论指数 b ≈ -0.25，聚合失败率大致按 $k^{-0.25}$ 下降。代码里采样写为 `p = scale * u ** (1.0 / alpha)`，这是逆变换采样：u 服从 (0, 1) 均匀分布，$u^{1/\alpha}$ 的分布函数是 $F(x) = x^\alpha$，正是 Kumaraswamy(alpha, 1) 分布，其密度在 0 附近正比于 $x^{\alpha-1}$。


In [ ]:
ks = np.logspace(0, 3, 80)                     # k = 1 .. 1000

p_hard = 0.01                                  # 单道"难"题的 p@1
fail_single = (1 - p_hard) ** ks               # 单题失败率：指数衰减

p1 = make_benchmark()
fail_agg = np.mean((1 - p1) ** ks[:, None], axis=1)

def loglog_r2(ks, fail):
    """log(fail) 关于 log(ks) 线性回归的 R^2，衡量幂律拟合程度。"""
    x = np.log(ks)
    y = np.log(fail)
    slope, intercept = np.polyfit(x, y, 1)
    pred = slope * x + intercept
    return 1 - np.sum((y - pred) ** 2) / np.sum((y - y.mean()) ** 2)

print("单题失败率 k=1 -> k=1000 下降 %.0f 倍" % (fail_single[0] / fail_single[-1]))
print("聚合失败率 k=1 -> k=1000 下降 %.1f 倍" % (fail_agg[0] / fail_agg[-1]))
print("聚合失败率 log-log 拟合 R^2: %.4f" % loglog_r2(ks, fail_agg))

rng = np.random.default_rng(1)
p_unif = rng.uniform(0.2, 0.4, 200)            # 无重左尾的 p@1 分布
fail_unif = np.mean((1 - p_unif) ** ks[:, None], axis=1)
print("无重尾(均匀 0.2~0.4)聚合失败率 R^2: %.4f" % loglog_r2(ks, fail_unif))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].loglog(ks, fail_single, label="single problem (exp)")
axes[0].loglog(ks, fail_agg, ".-", label="aggregated (power law)")
axes[0].set_xlabel("k")
axes[0].set_ylabel("failure rate")
axes[0].set_title("Exponential vs power law")
axes[0].legend()
axes[1].loglog(ks, fail_agg, ".-", label="heavy tail p@1")
axes[1].loglog(ks, fail_unif, ".-", label="uniform p@1")
axes[1].set_xlabel("k")
axes[1].set_ylabel("failure rate")
axes[1].set_title("Heavy tail makes the power law")
axes[1].legend()
plt.tight_layout()
plt.show()
print("关键观察：单题失败率陡降，聚合失败率在 log-log 上近似直线；"
      "去掉重尾分布后直线形态消失。")

## 3. 从采样到投票：self-consistency 与 best-of-n

覆盖率回答"模型能不能解出这道题"，最终答对多少还取决于怎么从候选里挑一个，也就是 precision。最简单的选择器有两个。第一个是多数投票：对每个问题生成 k 个候选，取出现次数最多的答案，这是 self-consistency（Wang 等人）的做法。多数投票不需要验证器，但它要求正确解成为众数；自由形式答案几乎不重合时，正确解至少要出现两次。第二个是 best-of-n：给每个候选打分，取分数最高的。Oracle 验证器永远挑得到正确解，真实验证器会有噪声。

论文里有一个反直觉的数字：MATH 上覆盖率提升到 95% 以上，多数投票却只从 40.50% 提升到 41.41%，reward model 加 best-of-n 也在约 100 个样本处封顶。覆盖率与真实成功率之间的 gap 随样本数扩大。我们用模拟重现这三条曲线：候选里有多类错误答案，错误类的扎堆会带偏多数投票。



### 多数投票被错误扎堆带偏的机制

多数投票对每个问题取出现次数最多的答案。要让多数投票答对，正确解必须严格成为众数。看一个极小的例子，某道题的 5 个候选是：

```text
✓   ✗₁   ✗₁   ✗₂   ✗₃
```

正确答案 ✓ 只出现一次。错误答案里 ✗₁ 出现两次，是众数，多数投票选 ✗₁，答错。注意这道题的覆盖率是 100%，候选里确实有正确解，Oracle 验证器能挑中 ✓。同一个 5 个候选，多数投票失败而 Oracle best-of-n 成功，差别全在选择器。

错误答案扎堆的原因在于 LLM 对同一道题犯的错误不是均匀散布的，而是集中在少数几条"看起来合理的错误路径"上。一道计算题最容易出错的地方往往只有两三处，于是错误样本堆进少数几个类别，某个类别的计数很容易超过正确解。正确解只有一种写法，每个正确样本都贡献给同一个类别；错误类别却可以各自积累，扎堆时单个错误类别的计数就能压过正确类别。

best-of-n 的收益可以分解成两个条件：候选里至少有一个正确的（覆盖率），且选择器恰好挑中它（精度）。把两个条件当作近似独立的事件相乘，得到

$$\text{accuracy} \approx \text{coverage} \times \rho$$

其中 $\rho$ 是验证器的精度。Oracle 验证器 $\rho = 1$，正确率等于覆盖率；精度 0.55 的弱验证器，正确率只有覆盖率的一半左右。代码里 `weak_bon = coverage * precision` 就是这个分解。多数投票没有验证器，它的"精度"由错误分布决定：错误扎堆越严重，正确解成为众数的比例越低。这就是论文里覆盖率升到 95% 以上、多数投票却只从 40.50% 升到 41.41% 的原因。

另外注意代码里多数投票用 `counts[0] > counts[1:].max()` 判断，要求正确类别严格多于任何单个错误类别。即使正确解出现两次、某个错误类别也出现两次，多数投票同样失败。


In [ ]:
W = 50                                       # 错误答案类别数
pool = 10000                                 # 每道题预生成的样本池
P = p1.shape[0]
rng = np.random.default_rng(5)

correct = rng.random((P, pool)) < p1[:, None]       # 每个样本是否正确
wrong = rng.integers(1, W + 1, size=(P, pool))      # 错误样本的类别
A = np.where(correct, 0, wrong)                     # 0 表示正确类

def majority_accuracy(A, k):
    """用前 k 个样本做多数投票，返回基准上的答对率。

    每道题统计各类别出现次数，正确类 0 严格多于其他类才算对。
    """
    hits = 0
    for row in A:
        counts = np.bincount(row[:k], minlength=W + 1)
        if counts[0] > counts[1:].max():
            hits += 1
    return hits / A.shape[0]

ks = np.unique(np.logspace(0, np.log10(5000), 40).astype(int))
coverage = np.array([coverage_at_k(p1, k).mean() for k in ks])
maj = np.array([majority_accuracy(A, k) for k in ks])
precision = 0.55                             # 固定精度的验证器
weak_bon = coverage * precision              # best-of-n = coverage x precision

for k, c, m, w in zip(ks[::5], coverage[::5], maj[::5], weak_bon[::5]):
    print("k=%5d  coverage=%.3f  majority=%.3f  weak-bo=%.3f" % (k, c, m, w))

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogx(ks, coverage, label="coverage (oracle)")
ax.semilogx(ks, weak_bon, label="best-of-n, precision 0.55")
ax.semilogx(ks, maj, label="majority vote")
ax.set_xlabel("k (samples per problem)")
ax.set_ylabel("accuracy")
ax.set_title("Coverage vs realized accuracy")
ax.legend()
plt.show()

In [ ]:
def gap_at(k_target, xs, ys):
    """在 log-spaced 的 ks 里找最接近 k_target 的位置，返回两条曲线的差。"""
    i = int(np.argmin(np.abs(ks - k_target)))
    return xs[i] - ys[i]

print("k=100  时 coverage - majority = %.3f" % gap_at(100, coverage, maj))
print("k=5000 时 coverage - majority = %.3f" % gap_at(5000, coverage, maj))
print("k=5000 时：coverage %.3f，weak best-of-n %.3f，majority %.3f"
      % (coverage[-1], weak_bon[-1], maj[-1]))
print("关键观察：coverage 逼近 1，majority 明显落后，gap 随 k 扩大；"
      "best-of-n 的收益 = coverage x precision，验证器的 precision 决定上限。")

## 4. Compute-optimal 缩放：Snell 定律

重复采样把所有预算花在同一个选择器上。Snell 等人把问题推进了一步：给定一笔固定的推理算力预算，不同难度的题目应该用不同方法。容易的题适合顺序修订（revision）——先给一个草稿，再顺着错误逐步修改；难的题适合并行重采样——让多个独立候选互相竞争。把预算按题目难度分配，就是 compute-optimal 缩放，论文报告同等精度下比均匀 best-of-n 少花约 4 倍算力。

难度的划分需要一个可操作的标准。论文用 base LLM 对每道题采 2048 个样本估出 pass@1，按 5 分位数切成 5 个难度档。我们沿用这个做法：把合成基准按 $p_i$ 升序排好，均分成 5 档，档 1 最难、档 5 最易。我们用一套简化的修订模型模拟"并行条数 × 顺序步数"的预算拆分，观察各档的最优拆分方式。



### 预算拆分的两个方向

compute-optimal 实验把一笔预算同时拆成两个方向：并行条数，即同时跑几条独立的链；顺序步数，即每条链内部做多少次修订。代码用参数 t 控制拆分：每条链的顺序步数 $N_{seq} = \text{budget}^t$，并行条数 $N_{par} = \text{budget} // N_{seq}$，两者乘积不超过预算。

用 budget = 16 手算三种情况。t = 0 时 $N_{seq} = 16^0 = 1$，$N_{par} = 16$，即 16 条一步完成的独立尝试，等价于纯重复采样。t = 1 时 $N_{seq} = 16$，$N_{par} = 1$，即一条链连修 16 步，等价于纯顺序修订。t = 0.5 时 $N_{seq} = 16^{0.5} = 4$，$N_{par} = 4$，即 4 条链各修 4 步，总预算恰好 4 × 4 = 16。t 从 0 滑到 1，就是把算力从广撒网挪向深挖掘。

难题偏并行、易题偏顺序的原因在修订模型 `revision_gain` 里：p@1 达到 0.03 的题，每一步修订有 3p 的概率修复成功（上限 0.9）；p@1 低于 0.03 的题，修订增益为 0，一连串修订只会绕着原来的错误打转，不产生新信息。

对难题（p@1 很低），增益为 0 时链的成功率恒等于 p，加长顺序步数毫无作用，唯一的提升途径是增加并行条数，让多个独立尝试互相竞争，所以最优 t 偏向 0。对易题，每次修订都显著提高成功率，把预算集中到一条链上连续修订，比分散成多条并行链更划算，最优 t 偏向 1。这解释了为什么档 1（最难）的最优 t 接近 0，档 5（最易）的最优 t 接近 1。


In [ ]:
def revision_gain(p, thresh=0.03, mult=3.0, cap=0.9):
    """修订的有效性：容易题能修，难题修不动。

    修订模型只在容易的修正上学过，对 p@1 很低的难题，
    一连串修订往往在原错误上打转，不产生新信息。
    """
    gain = np.where(p >= thresh, mult * p, 0.0)
    return np.minimum(gain, cap)

def chain_success(p, s):
    """长度为 s 的顺序修订链的通过概率。

    第一步以 p 做对；失败后每步以增益 revision_gain(p) 修正。
    """
    gain = revision_gain(p)
    return 1 - (1 - p) * (1 - gain) ** (s - 1)

def budget_split_success(p, t, budget=16):
    """把预算拆成并行与顺序：N_seq = budget**t 个顺序步、N_par 条并行链。

    总预算 = N_par * N_seq = budget。返回任一链成功的通过率。
    """
    n_seq = max(1, int(round(budget ** t)))
    n_par = budget // n_seq
    single = chain_success(p, n_seq)
    return 1 - (1 - single) ** n_par

order = np.argsort(p1)                       # 按 p@1 升序，最难在前
bins = np.array_split(order, 5)              # 切成 5 个难度档，档 1 最难
bin_p = [p1[i] for i in bins]                # 每档的 p@1 数组
ts = np.linspace(0, 1, 17)                   # t=0 全并行，t=1 全顺序

best_t = []
for q in bin_p:
    vals = [budget_split_success(q, t).mean() for t in ts]
    best_t.append(ts[int(np.argmax(vals))])
for bi, t in enumerate(best_t):
    print("bin %d 最优 t = %.2f" % (bi + 1, t))
print("趋势：最难题偏并行(t->0)，易题偏顺序(t->1)")

In [ ]:
success = np.array([[budget_split_success(q, t).mean() for t in ts] for q in bin_p])
rel = success / success.max(axis=1, keepdims=True)     # 每档按自身最优归一化

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
im = axes[0].imshow(rel, aspect="auto", origin="lower", cmap="viridis",
                    extent=[1, 5, ts[0], ts[-1]])
for bi, t in enumerate(best_t):
    axes[0].plot(bi + 1, t, "o", ms=6, mfc="white", mec="k")
axes[0].set_xlabel("difficulty bin (1 = hardest)")
axes[0].set_ylabel("t (1 = all sequential)")
axes[0].set_title("Optimal budget split (per-bin normalized)")
plt.colorbar(im, ax=axes[0])

for label, j in [("hardest", 0), ("middle", 2), ("easiest", 4)]:
    axes[1].plot(ts, success[j], label=label)
axes[1].set_xlabel("t (1 = all sequential)")
axes[1].set_ylabel("success rate")
axes[1].set_title("Success vs split, by difficulty")
axes[1].legend()
plt.tight_layout()
plt.show()

total = [budget_split_success(p1, t).mean() for t in ts]
t_unif = ts[int(np.argmax(total))]
acc_co = np.mean([budget_split_success(q, bt).mean()
                  for q, bt in zip(bin_p, best_t)])
print("统一分配（所有题共用一个 t）：最佳 t=%.2f，成功率 %.3f" % (t_unif, np.max(total)))
print("compute-optimal（每档一个 t）：成功率 %.3f" % acc_co)

def uniform_succ(budget):
    """统一分配下，给定预算能达到的最佳成功率。"""
    return max(budget_split_success(p1, t, budget=budget).mean() for t in ts)

for b in (16, 24, 32):
    print("统一分配预算 %3d -> 成功率 %.3f" % (b, uniform_succ(b)))
print("关键观察：同样 16 单位预算，compute-optimal 达到 %.3f；"
      "统一分配要到 24 单位预算才追平，约 1.5 倍算力差距。" % acc_co)

还有一个问题：test-time compute 与训练算力之间如何兑换。Snell 的答案是它取决于推理负载比例 $R = D_{infer}/D_{train}$。训练算力约 $6ND_{train}$，推理算力约 $2ND_{infer}$。把模型参数放大 M 倍，总算力变成 M 倍的训练加推理；用小模型加 test-time compute 匹配这笔总预算时，可用的采样次数变为

$$S = M + 3\,\frac{D_{train}}{D_{infer}}(M-1)$$

取 M = 14，R 取 0.16（自举式训练，推理负载低）、0.79（典型负载）、22（大规模部署，推理负载高）时，S 分别约 258、63、16。我们画出小模型的采样曲线，把"14 倍大模型贪心解码"的精度当作一条水平线，看三个 R 下的可用样本数能把小模型推到线上方还是下方。



### S 公式的算力推导

Snell 的结论建立在一笔算力账上。训练一个大模型大致需要 $6ND_{train}$ 次浮点运算：每个参数、每个训练 token 约 6 次，一次前向加两次反向，是自动微分的标准常数。推理时每个参数、每个输出 token 约 2 次，只有前向。记推理负载为 $D_{infer}$，大模型的总算力是

$$\text{total} = 6ND_{train} + 2ND_{infer}$$

把这笔总算力换一种花法：训练一个参数少 M 倍的模型 $N' = N/M$，其余算力全部投入重复采样。训练花费 $6(N/M)D_{train}$，剩余算力为

$$6ND_{train}\left(1 - \frac{1}{M}\right) + 2ND_{infer} = 2ND_{infer}\left[3\,\frac{D_{train}}{D_{infer}}\left(1 - \frac{1}{M}\right) + 1\right]$$

小模型采一个样本的成本是 $2(N/M)D_{infer}$，参数少 M 倍，每个输出 token 只花 2N/M 次运算。剩余算力除以单样本成本，得到可采的样本数

$$S = M + 3\,\frac{D_{train}}{D_{infer}}(M-1)$$

式子里的常数 3 来自训练与推理每 token 算力之比 6/2。$D_{train}/D_{infer}$ 就是 1/R，R 越小（推理负载低，比如自举式训练）这个比值越大，可采样本越多，test-time 越划算；R 越大（部署负载高）比值越小，剩余算力越少，扩训练更划算。取 M = 14、R = 0.16，$S = 14 + 3\times 13/0.16 = 14 + 243.75 \approx 258$，与代码输出的 258 一致。


In [ ]:
p_small = make_benchmark(alpha=0.8, scale=0.5, seed=11)    # 一组"小模型"题目
M = 14                                        # 大模型是 14 倍参数
m = 20                                        # 大模型 greedy 等价于小模型采 20 次
big_acc = (1 - (1 - p_small) ** m).mean()

S = np.arange(1, 301)
small_acc = np.array([(1 - (1 - p_small) ** s).mean() for s in S])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(S, small_acc, label="small model + repeated sampling")
ax.axhline(big_acc, color="gray", ls="--", label="14x big model (greedy)")
for R in (0.16, 0.79, 22.0):
    s_star = M + 3 * (M - 1) / R              # FLOPs-matched 的可用样本数
    acc = (1 - (1 - p_small) ** s_star).mean()
    winner = "test-time" if acc > big_acc else "pretrain"
    ax.plot(s_star, acc, "o")
    ax.annotate("R=%.2f -> %s" % (R, winner), (s_star, acc),
                textcoords="offset points", xytext=(0, 8), fontsize=9)
ax.set_xscale("log")
ax.set_xlabel("samples per problem")
ax.set_ylabel("accuracy")
ax.set_title("Small model + test-time vs 14x pretrain")
ax.legend()
plt.show()

print("R=0.16（自举）-> 可用样本 %.0f，test-time 占优" % (M + 3 * 13 / 0.16))
print("R=0.79（典型）-> 可用样本 %.0f，test-time 占优" % (M + 3 * 13 / 0.79))
print("R=22.0（部署）-> 可用样本 %.0f，pretraining 占优" % (M + 3 * 13 / 22.0))
print("关键观察：R 小（自举/推理负载低）时 test-time 更划算，R 大（部署）时扩训练更划算。")

## 5. 方法组合与架构搜索：Archon

前面几节的方法各有适用场景：重复采样对覆盖率有效，顺序修订对易题有效，best-of-n 依赖验证器。Archon 观察到没有哪个单一技术在所有任务上最优，于是把这些技术组织成分层的 LLM 系统，再用自动搜索找出最优组合。系统的每个组件都是 text-to-text 操作，没有可训练的权重：Generator 生成候选，Fuser 把多个候选合并，Ranker 两两比较排序，Critic 先列优缺点再交给排序与融合，Verifier 先给推理再给判定，Unit-Test 生成器产出测试语句并打分。

结构有若干硬规则：Generator 只能放在第一层，每层只放一类组件，Critic 必须在 Ranker 或 Fuser 之前，最后一层输出第一个字符串。去掉无效配置后，搜索空间有 9576 个配置，论文用贝叶斯优化在约两成数据上搜索，最佳架构平均超过当时 frontier 模型 15.1%。我们在一个 96 配置的简化空间里，从零实现网格搜索与随机搜索，观察最优架构是否随任务变化。



### 网格搜索与随机搜索的差异

96 个配置意味着最直接的做法是全部试一遍。代码里的 `grid_best` 遍历全部 96 个配置，每个配置跑一次合成评估，取最高分，这是网格搜索（grid search）。`random_search_trace` 换一种策略：不放回地随机挑 40 个配置，记录"到目前为止见过的最好分"随评估次数上升的曲线。

对比的结果是：随机 40 次评估就能接近网格最优。这不是巧合。合成评估器的分数变化平缓，大多数配置的性能落在相近的水平，真正的最优配置只比次优略高一点点。面对这种响应面平滑的问题，随机搜索不需要遍历所有组合，先随机探一圈就能锁定峰值的大致区域。

真实的 Archon 配置空间有 9576 个配置，逐一遍历成本太高，论文因此改用贝叶斯优化：先随机采样一批配置得到分数，用回归模型拟合"配置到分数"的曲面，再在预测分数最高的位置加密采样，如此迭代，把评估预算压到约两成数据以内。随机搜索是贝叶斯优化的起点，本节从零实现了网格与随机两种搜索，用来对照它们的性价比。


In [ ]:
configs = []
for top_k in (2, 4, 6, 8):
    for layers in (1, 2, 3):
        for critic in (0, 1):
            for verifier in (0, 1):
                for unit_test in (0, 1):
                    configs.append(dict(top_k=top_k, layers=layers, critic=critic,
                                        verifier=verifier, unit_test=unit_test))
print("配置总数:", len(configs))

rng_eval = np.random.default_rng(5)
score_table = {}

def architecture_accuracy(cfg, task):
    """合成评估器：读缓存分数，没有则计算并缓存。

    cfg：配置字典；task：'instruct' 或 'code'。
    融合层对指令跟随有用、单测对代码有用、验证器对推理有用。
    """
    key = (task, cfg["top_k"], cfg["layers"], cfg["critic"],
           cfg["verifier"], cfg["unit_test"])
    if key in score_table:
        return score_table[key]
    acc = 0.55 if task == "instruct" else 0.30
    acc += 0.008 * cfg["top_k"]
    acc += 0.030 * cfg["layers"] * (1.0 if task == "instruct" else 0.35)
    acc += 0.035 * cfg["critic"] * (1.0 if task == "instruct" else 0.10)
    acc += 0.025 * cfg["verifier"] * (1.0 if task == "instruct" else -0.20)
    acc += 0.090 * cfg["unit_test"] if task == "code" else -0.025 * cfg["unit_test"]
    acc += rng_eval.normal(0, 0.012)          # 评估噪声
    score_table[key] = min(acc, 1.0)
    return score_table[key]

def grid_best(configs, task):
    """遍历全部配置，返回（最高分，配置）。"""
    scores = [architecture_accuracy(c, task) for c in configs]
    i = int(np.argmax(scores))
    return scores[i], configs[i]

def random_search_trace(configs, task, rng, n=40):
    """不放回随机采 n 个配置，记录每一步的当前最优分。"""
    order = rng.permutation(len(configs))
    best = []
    for step in range(min(n, len(configs))):
        sc = architecture_accuracy(configs[order[step]], task)
        best.append(sc if step == 0 else max(best[-1], sc))
    return np.array(best)

for task in ("instruct", "code"):
    score = grid_best(configs, task)[0]
    trace = random_search_trace(configs, task, np.random.default_rng(9))
    print("%s: grid 最优 %.3f，随机 40 次评估达到 %.3f（%.1f%%）"
          % (task, score, trace[-1], 100 * trace[-1] / score))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for task, color in [("instruct", "C0"), ("code", "C3")]:
    grid_score = grid_best(configs, task)[0]
    trace = random_search_trace(configs, task, np.random.default_rng(9))
    ax.plot(range(1, len(trace) + 1), trace, "-o", ms=3, color=color,
            label="%s random" % task)
    ax.axhline(grid_score, color=color, ls="--", label="%s grid best" % task)
ax.set_xlabel("number of evaluations")
ax.set_ylabel("best accuracy found")
ax.set_title("Architecture search: random vs grid")
ax.legend(fontsize=8)
plt.show()

instruct_cfg = grid_best(configs, "instruct")[1]
code_cfg = grid_best(configs, "code")[1]
print("指令跟随任务的最优配置:", instruct_cfg)
print("代码任务的最优配置:", code_cfg)
print("关键观察：两个任务的最优配置不同，没有通吃的单一架构，"
      "这正是自动搜索的动机。")

In [ ]:
import sys, os
_root = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_root, 'llm_client.py')):
    _root = os.path.dirname(_root)
    if _root == os.path.dirname(_root):
        break
if _root not in sys.path:
    sys.path.insert(0, _root)
from llm_client import get_llm
client = get_llm()
import os
mock_requested = bool(os.environ.get("LLM_MOCK")
                      or os.environ.get("AGENT_LLM_MOCK"))
effective_mock = client.is_mock or mock_requested
print("LLM 客户端就绪，有效模式:",
      "mock（确定性占位）" if effective_mock else "real API")

In [ ]:
import re

def generate_candidates(client, question, n=3):
    """用 n 种提示模板让模型各自作答，返回候选答案字符串。"""
    templates = [
        "请计算 %s，只输出最终答案。",
        "%s 等于多少？只输出数字。",
        "求解：%s。只输出最终答案。",
    ]
    candidates = []
    for i in range(n):
        reply = client.chat([{"role": "user", "content": templates[i] % question}])
        nums = re.findall(r"\d+", reply)
        candidates.append(nums[0] if nums else "N/A")
    return candidates

def majority(answers):
    """多数投票：返回出现次数最多的答案。"""
    votes = {}
    for a in answers:
        votes[a] = votes.get(a, 0) + 1
    return max(votes, key=votes.get)

question = "15 加 27"
mock_requested = bool(os.environ.get("LLM_MOCK")
                      or os.environ.get("AGENT_LLM_MOCK"))
if mock_requested or not getattr(client, "api_key", ""):
    client = get_llm(force_mock=True)        # 离线验证一律走 mock，避免真实调用
try:
    answers = generate_candidates(client, question, n=3)
except RuntimeError:
    client = get_llm(force_mock=True)
    answers = generate_candidates(client, question, n=3)
    print("未检测到可用 API key，已切换到 mock 模式")

print("三个候选答案:", answers)
print("多数投票结果:", majority(answers), "（正确答案 42）")
if client.is_mock:
    print("说明：mock 模式下回复为确定性占位；真实 API 下温度采样会给候选带来多样性。")

## 小结

这一讲沿着"一笔推理算力预算如何分配"这条决策链走了一遍：

- [ ] 训练完成后仍可花推理算力换取能力，所有 test-time 方法都落在 proposer 与 verifier 两个旋钮上
- [ ] 重复采样把覆盖率推高，pass@k 需要无偏估计，直接统计会系统性高估
- [ ] 覆盖率呈幂律 $\exp(a\,k^b)$，用 log-log 线性回归即可拟合
- [ ] 单道题失败率指数衰减，聚合到整个基准变成幂律；去掉重尾分布，幂律消失
- [ ] 多数投票与 best-of-n 的真实收益由 precision 决定，无可靠验证器时收益封顶
- [ ] 固定预算下按题目难度分配并行与顺序算力，compute-optimal 优于统一分配
- [ ] 小模型加 test-time compute 与更大模型谁的算力更值，由推理负载比例 R 决定
- [ ] 多种 test-time 技术可组合成分层系统，用自动搜索替代手工搭架构


## 作业

> 可以让 AI 帮忙解释思路，但不建议直接让 AI "做完这道题"。

**作业 1：pass@k 无偏估计**

补全下面的函数，完成 pass@k 的数值稳定形式。已给出 N = 20、C = 5 的验证，预期 pass@5 ≈ 0.806，pass@20 = 1。

```python
def estimate_pass_at_k(num_correct, num_samples, k):
    if num_correct == 0:
        return 0.0
    denom = np.arange(num_samples - num_correct + 1, num_samples + 1)
    return 1.0 - np.prod(____)      # 补全

assert abs(estimate_pass_at_k(5, 20, 5) - 0.8063) < 1e-3
assert abs(estimate_pass_at_k(5, 20, 20) - 1.0) < 1e-12
print("无偏估计实现正确：k 越接近 N，朴素估计的高估越明显。")
```

小提示：`np.prod(1 - k / denom)` 就是组合数之比 $\binom{N-C}{k}/\binom{N}{k}$，二者在 k > N-C 时都等于 0。

**作业 2：从重尾分布推出幂律指数**

给定合成 p@1（Kumaraswamy(0.4, 1) 乘 0.15），模拟聚合覆盖率，在 log-log 上拟合出指数 b，并与左尾理论值 -alpha 对照。

```python
p = make_benchmark(alpha=0.4, scale=0.15, seed=3)
ks = np.logspace(1, 2.7, 40).astype(int)
covs = np.array([(1 - (1 - p) ** k).mean() for k in ks])
slope, intercept = np.polyfit(np.log(ks), np.log(-np.log(covs)), 1)
b = ____                                    # 补全：斜率即 b

assert abs(b - (-0.4)) < 0.25
print("聚合幂律指数 b 由 p@1 分布左尾指数决定：理论值 -0.4，拟合值 %.3f。" % b)
```

小提示：Kumaraswamy(alpha, 1) 的密度在 0 附近正比于 $p^{\alpha-1}$，定理 3.1 给出聚合失败率按 $k^{-\alpha}$ 缩放。有限 k 区间上拟合值会略偏离渐近值，断言因此放宽到 0.25。


**作业 3：多数投票的 plateau**

合成数据上，多数投票需要正确类严格成为众数。补全条件，验证 majority 的提升明显小于 coverage。

```python
W = 50
pool = 10000
p = make_benchmark(seed=5)
P = p.shape[0]
rng = np.random.default_rng(5)
correct = rng.random((P, pool)) < p[:, None]
wrong = rng.integers(1, W + 1, size=(P, pool))
A = np.where(correct, 0, wrong)

def majority_accuracy(A, k):
    hits = 0
    for row in A:
        counts = np.bincount(row[:k], minlength=W + 1)
        if counts[0] > ____:                 # 补全：正确类严格多于其他类
            hits += 1
    return hits / A.shape[0]

c_100 = (1 - (1 - p) ** 100).mean()
c_5000 = (1 - (1 - p) ** 5000).mean()
m_100 = majority_accuracy(A, 100)
m_5000 = majority_accuracy(A, 5000)
assert (m_5000 - m_100) < (c_5000 - c_100)
print("k 从 100 到 5000：coverage 提升 %.3f，majority 只提升 %.3f——"
      "多数投票对难题无能为力，gap 随样本数扩大。" % (c_5000 - c_100, m_5000 - m_100))
```

小提示：每类错误答案被"扎堆"时，正确类必须严格多于任何一类才能成为众数，即 `counts[0] > counts[1:].max()`。

## 参考资料

- [Large Language Monkeys: Scaling Inference Compute with Repeated Sampling](https://arxiv.org/abs/2407.21787)（Brown 等人，2024）— 重复采样覆盖率的实证与幂律，coverage/precision 两轴的出处
- [Scaling LLM Test-Time Compute Optimally can be More Effective than Scaling Model Parameters](https://arxiv.org/abs/2408.03314)（Snell 等人，2024）— compute-optimal 缩放：按难度分配 test-time 算力，revision 加 PRM 搜索
- [Archon: An Architecture Search Framework for Inference-Time Techniques](https://arxiv.org/abs/2409.15254)（Saad-Falcon 等人，ICML 2025）— 分层 LLM 系统加贝叶斯优化架构搜索；注意正确 ID 是 2409.15254
- [How Do Large Language Monkeys Get Their Power (Laws)?](https://arxiv.org/abs/2502.17578)（Schaeffer 等人，ICML 2025）— 单题指数衰减加重尾 p@1 分布推出聚合幂律；注意正确 ID 是 2502.17578
- [Evaluating Large Language Models Trained on Code](https://arxiv.org/abs/2107.03374)（Chen 等人，2021）— pass@k 无偏估计器的出处
- [Self-Consistency Improves Chain of Thought Reasoning](https://arxiv.org/abs/2203.11171)（Wang 等人，2023）— 多数投票 / self-consistency，与 precision 的讨论直接相关
- [Training Verifiers to Solve Math Word Problems](https://arxiv.org/abs/2110.14168)（Cobbe 等人，2021）— GSM8K 与最早验证器训练
- [Let's Verify Step by Step](https://arxiv.org/abs/2305.20050)（Lightman 等人，2023）— PRM 训练与 MATH 难度分档，Snell 论文沿用其数据划分
- [Competition-Level Code Generation with AlphaCode](https://arxiv.org/abs/2203.07814)（Li 等人，2022）— 大规模重复采样的先驱，CodeContests 数据集出处
- [Beyond Chinchilla-Optimal: Accounting for Inference in LM Scaling Laws](https://arxiv.org/abs/2401.00448)（Sardana 与 Frankle，2023）— 把推理 FLOPs 计入缩放定律，Snell 论文 FLOPs 公式的依据